In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import cv2
# 1. Load alignment matrix M
# Note: the official CSV usually has no header, or the first row is data
mat_df = pd.read_csv("Xenium_Prime_Human_Prostate_FFPE_he_imagealignment.csv", header=None)
M = mat_df.values

# 2. Compute inverse matrix M_inv
M_inv = np.linalg.inv(M)

# 3. Load Xenium nucleus boundaries data (micron units)
# Here we use nucleus_boundaries.csv.gz as an example
boundaries = pd.read_csv("nucleus_boundaries.csv.gz")


In [2]:
boundaries

,cell_id,vertex_x,vertex_y,label_id
0,aaaamcnn-1,141.3125,2475.2000,1
1,aaaamcnn-1,140.6750,2475.4126,1
2,aaaamcnn-1,138.9750,2476.2625,1
3,aaaamcnn-1,138.1250,2477.1125,1
4,aaaamcnn-1,137.7000,2478.1750,1
...,...,...,...,...
4695626,oiohaagd-1,4880.7000,3249.5500,191544
4695627,oiohaagd-1,4880.2750,3249.3376,191544
4695628,oiohaagd-1,4879.8500,3249.3376,191544
4695629,oiohaagd-1,4879.6377,3249.1250,191544


In [3]:
# 4. Execute coordinate conversion
# Step 1: microns -> Xenium internal pixels (divide by 0.2125)
boundaries['x_xe_px'] = boundaries['vertex_x'] / 0.2125
boundaries['y_xe_px'] = boundaries['vertex_y'] / 0.2125

# Step 2: build homogeneous coordinate matrix [X, Y, 1]
xe_pixels = boundaries[['x_xe_px', 'y_xe_px']].values
ones = np.ones((xe_pixels.shape[0], 1))
xe_homogeneous = np.hstack([xe_pixels, ones]) # becomes [N, 3] matrix

In [4]:
he_pixels_homogeneous = (M_inv @ xe_homogeneous.T).T

# 5. Save results back to DataFrame
boundaries['he_pixel_x'] = he_pixels_homogeneous[:, 0]
boundaries['he_pixel_y'] = he_pixels_homogeneous[:, 1]

# View first few rows
print(boundaries[['cell_id', 'vertex_x', 'vertex_y', 'he_pixel_x', 'he_pixel_y']].head())

      cell_id  vertex_x   vertex_y    he_pixel_x   he_pixel_y
0  aaaamcnn-1  141.3125  2475.2000  15134.046213  6447.093366
1  aaaamcnn-1  140.6750  2475.4126  15133.246185  6444.772455
2  aaaamcnn-1  138.9750  2476.2625  15130.079127  6438.593756
3  aaaamcnn-1  138.1250  2477.1125  15126.942935  6435.520024
4  aaaamcnn-1  137.7000  2478.1750  15123.046117  6434.006580


In [5]:
# 1. Create a new DataFrame for saving, avoid modifying original boundaries data in memory
output_df = boundaries.copy()

# 2. Remove original micron coordinate columns (vertex_x, vertex_y)
output_df = output_df.drop(columns=['x_xe_px','y_xe_px','vertex_x', 'vertex_y'])

# 3. Rename H&E pixel coordinates to original column names
output_df = output_df.rename(columns={
    'he_pixel_x': 'vertex_x',
    'he_pixel_y': 'vertex_y'
})

# 4. Reorder columns to match the original file (optional, but recommended)
# Assume original column order is [cell_id, vertex_x, vertex_y, ...]
# Get all column names except vertex, rearrange
cols = list(output_df.columns)

In [6]:
# Define new filename
output_filename = "nucleus_boundaries_HE_coords.csv.gz"

print(f"Saving transformed coordinates to: {output_filename} ...")

# Save as .csv.gz format
# index=False ensures row indices are not saved
output_df.to_csv(output_filename, index=False, compression='gzip')

print("Save complete!")


Saving transformed coordinates to: nucleus_boundaries_HE_coords.csv.gz ...
Save complete!


In [7]:
he_image_path = "/sibcb1/chenluonanlab8/zoujiawei/Prostate/Xenium_Prime_Human_Prostate_FFPE_he_image.ome.tif"

In [ ]:
import tifffile
import zarr
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Prepare data (assumes boundaries already has he_pixel_x/y computed) ---
df = boundaries 
sample_idx = 5000
center_x = int(df.iloc[sample_idx]['he_pixel_x'])
center_y = int(df.iloc[sample_idx]['he_pixel_y'])

crop_size = 500
x_s = max(0, center_x - crop_size // 2)
y_s = max(0, center_y - crop_size // 2)

# --- 2. Use Zarr to open image (very efficient) ---
# aszarr=True creates a lightweight mapping without loading actual image data
store = tifffile.imread(he_image_path, aszarr=True)
z_img = zarr.open(store, mode='r')

# Xenium OME-TIFF is usually a pyramid structure
# z_img[0] is the highest resolution level
# Its shape is usually (Height, Width, Channels) or (Channels, Height, Width)
full_img = z_img[0]
print(f"Full image shape: {full_img.shape}")

# --- 3. Slice ROI ---
# Auto-handle dimension issues (for H&E, typically Y, X, C)
if full_img.ndim == 3:
    # Assume shape is (H, W, 3)
    roi = full_img[y_s : y_s + crop_size, x_s : x_s + crop_size, :]
else:
    # If only (H, W)
    roi = full_img[y_s : y_s + crop_size, x_s : x_s + crop_size]

# --- 4. Filter cells falling within this region ---
mask = (df['he_pixel_x'] >= x_s) & (df['he_pixel_x'] <= x_s + crop_size) & \
       (df['he_pixel_y'] >= y_s) & (df['he_pixel_y'] <= y_s + crop_size)
df_plot = df[mask].copy()

# Convert to local coordinates relative to ROI
df_plot['plot_x'] = df_plot['he_pixel_x'] - x_s
df_plot['plot_y'] = df_plot['he_pixel_y'] - y_s

# --- 5. Visualize ---
plt.figure(figsize=(10, 10))

# Check data type, scale if uint16
img_to_show = np.array(roi) # convert to regular numpy array for display
if img_to_show.dtype == np.uint16:
    img_to_show = (img_to_show / (img_to_show.max() / 255)).astype(np.uint8)

plt.imshow(img_to_show)

# Draw red dots
plt.scatter(df_plot['plot_x'], df_plot['plot_y'], 
            s=15, c='red', alpha=0.8, edgecolors='white', linewidths=0.5)

plt.title(f"H&E Alignment (Zarr Mode) Center: {center_x}, {center_y}")
plt.axis('off')
plt.show()
